In [1]:
from pathlib import Path
import os
import torch
import matplotlib.pyplot as plt
import matplotlib.animation as animation
import numpy as np

# Resolve project root robustly whether launched from repo root or main/.
cwd = Path.cwd()
if (cwd / 'config').exists():
    project_root = cwd
elif (cwd.parent / 'config').exists():
    project_root = cwd.parent
elif Path('/mnt/fastertalk/config').exists():
    project_root = Path('/mnt/fastertalk')
else:
    raise RuntimeError('Could not locate project root containing config/.')

os.chdir(project_root)
print('Project root:', Path.cwd())

from flame_model.FLAME import FLAMEModel
from renderer.renderer import Renderer
from pytorch3d.transforms import matrix_to_euler_angles

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
flame = FLAMEModel(n_shape=300, n_exp=50).to(device)
renderer = Renderer(render_full_head=True).to(device)
print('Renderer and FLAME ready on', device)

Project root: /mnt/fastertalk
Renderer and FLAME ready on cuda


/root/miniconda3/envs/fasttalk/lib/python3.11/site-packages/pytorch3d/io/obj_io.py:547: UserWarning: No mtl file provided
  warnings.warn("No mtl file provided")


In [2]:
from utils.config import load_flat_config
from models import get_model
from dataset.data_loader_joint_data_batched import get_dataloaders
import numpy as np

cfg = load_flat_config('config/talkinghead-1kh/stage1_style.yaml')
cfg.batch_size = 1
cfg.save_path = '/mnt/fastertalk/logs/stage1_style/checkpoints/epoch_300.pt'
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Style axis names (matches dataloader order)
STYLE_NAMES = ['lips_disp', 'lips_speed', 'forehead_disp', 'forehead_speed', 'eyes_disp', 'eyes_speed']
N_STYLE = len(STYLE_NAMES)

print('Device:', device)
print('Checkpoint:', cfg.save_path)
print('n_embed:', cfg.n_embed)
print('style_dim:', cfg.style_dim)
print('Style axes:', STYLE_NAMES)


Device: cuda
Checkpoint: /mnt/fastertalk/logs/stage1_style/checkpoints/epoch_300.pt
n_embed: 2048
style_dim: 32
Style axes: ['lips_disp', 'lips_speed', 'forehead_disp', 'forehead_speed', 'eyes_disp', 'eyes_speed']


In [ ]:
dls = get_dataloaders(cfg)
test_loader = dls['test']

def _resolve_checkpoint(path_cfg):
    path_cfg = Path(path_cfg)
    if path_cfg.is_file():
        return path_cfg
    if path_cfg.is_dir():
        ckpts = sorted(path_cfg.glob('epoch_*.pt'))
        return ckpts[-1] if ckpts else None
    return None

ckpt_file = _resolve_checkpoint(cfg.save_path)
state_dict = None
if ckpt_file is not None:
    ckpt = torch.load(ckpt_file, map_location=device)
    state_dict = ckpt['state_dict'] if isinstance(ckpt, dict) and 'state_dict' in ckpt else ckpt

model = get_model(cfg).to(device)
if state_dict is not None:
    missing, unexpected = model.load_state_dict(state_dict, strict=False)
    print('Loaded checkpoint:', ckpt_file)
    print('Missing keys:', len(missing), '| Unexpected keys:', len(unexpected))
else:
    print('No valid checkpoint found - using random weights.')


Loading data...
Data root: /mnt/Datasets/ARTalk_data
Audio path: /mnt/Datasets/ARTalk_data/wav
Vertices path: /mnt/Datasets/ARTalk_data/npz
Annotations root: /mnt/Datasets/ARTalk_data/annotations
Style disp stats loaded (z-score normalization enabled).
Train lines read: 5589
Test lines read: 621


 16%|█▋        | 1019/6210 [00:01<00:05, 895.56it/s]

In [ ]:
# --- Quick sanity check with GT style from the first test batch ---
model.eval()
batch = next(iter(test_loader))
blendshapes_in, blendshapes_tgt, mask, style = batch
blendshapes_in  = blendshapes_in.to(device)
blendshapes_tgt = blendshapes_tgt.to(device)
mask  = mask.to(device)
style = style.to(device)

with torch.no_grad():
    pred, qloss = model(blendshapes_in, mask, style=style)

valid = mask.unsqueeze(-1).float()
mse = (((pred[..., :-2] - blendshapes_tgt[..., :-2]) ** 2) * valid).sum() / \
      torch.clamp(valid.sum() * pred[..., :-2].shape[-1], min=1.0)

print('Input shape :', tuple(blendshapes_in.shape))
print('Pred shape  :', tuple(pred.shape))
print('Quant loss  :', float(qloss.mean().item()))
print('Masked MSE  :', float(mse.item()))
print('Style vector:', style[0].cpu().numpy().round(3))
print('Style axes  :', STYLE_NAMES)


In [ ]:
def get_vertices_from_blendshapes(expr, gpose, jaw, eyelids):
    expr_tensor = expr.to(device)
    gpose_tensor = gpose.to(device)
    jaw_tensor = jaw.to(device)
    _ = eyelids.to(device)
    
    target_shape_tensor = torch.zeros(expr_tensor.shape[0], 300, device=device)
    I = matrix_to_euler_angles(torch.cat([torch.eye(3, device=device)[None]], dim=0), 'XYZ')
    eye_r = I.clone().squeeze()
    eye_l = I.clone().squeeze()
    eyes = torch.cat([eye_r, eye_l], dim=0).expand(expr_tensor.shape[0], -1)
    pose = torch.cat([gpose_tensor, jaw_tensor], dim=-1)
    
    verts, _ = flame.forward(
        shape_params=target_shape_tensor,
        expression_params=expr_tensor,
        pose_params=pose,
        eye_pose_params=eyes,
    )
    return verts.detach()


def render_comparison(blendshapes_gt, blendshapes_pred, index, out_dir='demo/video_style'):
    expr_gt = blendshapes_gt[:, :50];   gpose_gt = blendshapes_gt[:, 50:53]
    jaw_gt  = blendshapes_gt[:, 53:56]; eyelids_gt = blendshapes_gt[:, 56:]
    expr_pr = blendshapes_pred[:, :50]; gpose_pr = blendshapes_pred[:, 50:53]
    jaw_pr  = blendshapes_pred[:, 53:56]; eyelids_pr = blendshapes_pred[:, 56:]

    verts_gt = get_vertices_from_blendshapes(expr_gt, gpose_gt, jaw_gt, eyelids_gt)
    verts_pr = get_vertices_from_blendshapes(expr_pr, gpose_pr, jaw_pr, eyelids_pr)

    cam = torch.tensor([5, 0, 0], dtype=torch.float32, device=device).unsqueeze(0)
    cam = cam.expand(verts_gt.shape[0], -1)
    frames_gt = renderer.forward(verts_gt, cam)['rendered_img']
    frames_pr = renderer.forward(verts_pr, cam)['rendered_img']

    os.makedirs(out_dir, exist_ok=True)
    video_file = os.path.join(out_dir, f'sample_{index:03d}.mp4')

    def update(frame_idx, gt_seq, pr_seq, ax):
        gt = gt_seq[frame_idx].detach().cpu().numpy().transpose(1, 2, 0)
        pr = pr_seq[frame_idx].detach().cpu().numpy().transpose(1, 2, 0)
        combined = np.concatenate([(gt * 255).astype(np.uint8), (pr * 255).astype(np.uint8)], axis=1)
        ax.clear(); ax.imshow(combined); ax.axis('off')

    fig, ax = plt.subplots(figsize=(10, 5))
    ani = animation.FuncAnimation(fig, update, frames=frames_gt.shape[0],
                                  fargs=(frames_gt, frames_pr, ax), interval=40)
    ani.save(video_file, writer='ffmpeg', fps=25)
    plt.close(fig)
    print('Saved video comparison to', video_file)


def evaluate_samples(model, data_loader, num_samples=3):
    model.eval()
    for i, batch in enumerate(data_loader):
        if i >= num_samples:
            break
        blendshapes_in, blendshapes_tgt, mask, style = batch   # 4-tuple
        blendshapes_in  = blendshapes_in.to(device)
        blendshapes_tgt = blendshapes_tgt.to(device)
        mask  = mask.to(device)
        style = style.to(device)

        with torch.no_grad():
            pred, qloss = model(blendshapes_in, mask, style=style)

        valid = mask.unsqueeze(-1).float()
        mse = (((pred[..., :-2] - blendshapes_tgt[..., :-2]) ** 2) * valid).sum() / \
              torch.clamp(valid.sum() * pred[..., :-2].shape[-1], min=1.0)

        print(f'sample={i}  quant={float(qloss.mean().item()):.6f}  mse={float(mse.item()):.6f}')
        print(f'         style={style[0].cpu().numpy().round(2)}')
        render_comparison(blendshapes_tgt.squeeze(0), pred.squeeze(0), i)


In [ ]:
# --- Style sweep: one side-by-side video per axis ---
# Each exported video shows all sweep values simultaneously:
#   [val=-2σ | val=-1σ | val=0 | val=+1σ | val=+2σ]
# One video per axis → 6 videos total in demo/video_style/sweep/.

SWEEP_VALUES = (-2.0, -1.0, 0.0, 1.0, 2.0)

def render_sweep_axis(model, blendshapes_in, mask, axis_idx, axis_name,
                      values=SWEEP_VALUES, out_dir='demo/video_style/sweep'):
    """Render one video with all sweep values side-by-side for a single axis."""
    model.eval()
    os.makedirs(out_dir, exist_ok=True)

    # --- Decode once per value, collect frame tensors ---
    all_frames = []   # list of (T, H, W, 3) uint8 arrays, one per value
    for val in values:
        style_vec = torch.zeros(1, N_STYLE, device=device)
        style_vec[0, axis_idx] = val
        with torch.no_grad():
            pred, _ = model(blendshapes_in, mask, style=style_vec)
        pred_seq = pred.squeeze(0)
        verts = get_vertices_from_blendshapes(
            pred_seq[:, :50], pred_seq[:, 50:53],
            pred_seq[:, 53:56], pred_seq[:, 56:]
        )
        cam    = torch.tensor([5, 0, 0], dtype=torch.float32, device=device).unsqueeze(0)
        cam    = cam.expand(verts.shape[0], -1)
        frames = renderer.forward(verts, cam)['rendered_img']   # (T, 3, H, W)  [0,1]
        frames_np = (frames.permute(0, 2, 3, 1).clamp(0, 1).cpu().numpy() * 255).astype(np.uint8)
        all_frames.append(frames_np)

    T = min(f.shape[0] for f in all_frames)
    H, W = all_frames[0].shape[1], all_frames[0].shape[2]
    n_vals = len(values)

    # --- Animate: concatenate horizontally, add per-panel label ---
    fig, axes = plt.subplots(1, n_vals, figsize=(4 * n_vals, 4.5),
                              gridspec_kw={'wspace': 0.02})
    ims = []
    for ax, frames_np, val in zip(axes, all_frames, values):
        ax.axis('off')
        ax.set_title(f'{val:+.0f}σ', fontsize=12, pad=3)
        im = ax.imshow(frames_np[0])
        ims.append(im)
    fig.suptitle(axis_name, fontsize=13, y=1.01)

    def update(t):
        for im, frames_np in zip(ims, all_frames):
            im.set_data(frames_np[min(t, frames_np.shape[0] - 1)])
        return ims

    ani = animation.FuncAnimation(fig, update, frames=T, interval=40, blit=True)
    out_path = os.path.join(out_dir, f'{axis_name}_sweep.mp4')
    ani.save(out_path, writer='ffmpeg', fps=25)
    plt.close(fig)
    print(f'  Saved → {out_path}')


# Use the test clip already loaded in cell 4
blendshapes_in_s = blendshapes_in
mask_s = mask

for i, name in enumerate(STYLE_NAMES):
    print(f'Sweeping axis {i}: {name}')
    render_sweep_axis(model, blendshapes_in_s, mask_s,
                      axis_idx=i, axis_name=name,
                      out_dir='demo/video_style/sweep')


In [ ]:
# Render a few test samples as input-vs-output videos.
evaluate_samples(model, test_loader, num_samples=3)